# Trinity THLP MC Test - Fixed
DeepMind AGI Hackathon 2026

In [ ]:
# Install dependencies
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

In [ ]:
# Imports
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
import glob
from dataclasses import dataclass

print("✅ Imports successful")

In [ ]:
# MC Answer schema
@dataclass
class MCAnswer:
    answer: str

print("✅ MC schema defined")

In [ ]:
# Download MC dataset
print("📥 Downloading MC dataset...")
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-thlp-mc',
    path='/kaggle/working/datasets/',
    unzip=True
)

csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'thlp_mc.csv' in f][0]

print(f"📂 Using: {csv_path}")

df = pd.read_csv(csv_path)
mc_df = df[df['question_type'] == 'mc'].copy()

# For testing: 50 items only
eval_df = mc_df.head(50)

print(f"📊 Loaded {len(eval_df)} MC questions")

In [ ]:
# MC matching function
def match_mc_answer(response: str, expected: str) -> bool:
    """MC format: compare first letter only (A, B, C, D)."""
    if not response or not expected:
        return False
    got = response.strip().upper()[0]
    exp = str(expected).strip().upper()[0]
    return got == exp

print("✅ MC matching function defined")

In [ ]:
# Inner MC task
@kbench.task(name="trinity_thlp_single_mc", store_task=False)
def thlp_single(llm, question, choices, expected_answer) -> bool:
    prompt = f"""{question}\n\n{choices}\n\nAnswer with ONLY option letter (A, B, C, or D)."""
    
    response = llm.prompt(prompt, schema=MCAnswer)
    
    return match_mc_answer(response.answer, expected_answer)

print("✅ Inner task registered")

In [ ]:
# Outer benchmark task
@kbench.task(
    name="Trinity Hippocampal Learning Probe (MC) 50-Item Test",
    description="MC benchmark for hippocampal learning (THLP) with single-letter answer matching. Tests 50 MC questions."
)
def thlp_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = thlp_single.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    
    results_df = runs.as_dataframe()
    valid = results_df[results_df["result"].notna()]
    
    if len(valid) == 0:
        kbench.assertions.assert_true(False, expectation="No valid results")
        return 0.0
    
    accuracy = float(valid["result"].mean())
    kbench.assertions.assert_true(
        True,
        expectation=f"THLP MC accuracy: {accuracy:.2%} ({len(valid)}/{len(eval_df)})"
    )
    
    return accuracy

print("✅ Outer benchmark task registered")

In [ ]:
# Run benchmark - FIXED VERSION
run = thlp_benchmark.run()  # kbench автоматически использует Claude
print(f"\\n🏆 Result: {run.result:.2%} ({len(eval_df)} items)")
print(f"\\nExpected: ~60-80% for good model, ~20% for random baseline")

%choose thlp_benchmark